# SOC-Graph Behavioral Detector Demo

This notebook demonstrates the current graph-first pipeline using an already trained behavioral detector artifact.

It does not train a model inside the notebook. Instead, it loads a saved model, runs inference on the sample PIDSMaker-style fixture, and shows the resulting alert payload.

In [1]:
from datetime import timedelta
from pathlib import Path
import json
from pprint import pprint

from soc_graph.data.dataset import build_datasets
from soc_graph.data.pidsmaker import load_events
from soc_graph.detection.subgraph import extract_candidate_subgraph
from soc_graph.model.io import load_detector
from soc_graph.model.train import detect_anomalies
from soc_graph.report.serialize import serialize_alert_subgraph

repo_root = Path.cwd()
if not (repo_root / 'artifacts').exists():
    repo_root = repo_root.parent

fixture_path = repo_root / 'tests/fixtures/sample_pidsmaker.csv'
model_path = repo_root / 'artifacts/models/behavioral_detector.json'
summary_path = repo_root / 'artifacts/models/behavioral_detector_summary.json'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary

{'input_csv': 'tests\\fixtures\\sample_pidsmaker.csv',
 'output_model': 'artifacts\\models\\behavioral_detector.json',
 'window_minutes': 15,
 'benign_ratio': 0.5,
 'threshold_k': 1.0,
 'training_summary': {'num_windows': 1,
  'mean_edges_per_window': 1,
  'learned_threshold': 3.4657359027997265,
  'benign_score_count': 1},
 'graph_stats': {'num_windows': 2.0,
  'mean_nodes_per_window': 2,
  'mean_edges_per_window': 1,
  'mean_edge_observations_per_window': 3},
 'train_windows': 1}

In [2]:
events = load_events(fixture_path)
snapshot_dataset, artifact_dataset = build_datasets(events, window=timedelta(minutes=15))

print(f'Loaded events: {len(events)}')
print(f'Window count: {len(snapshot_dataset)}')
print('Graph stats from training summary:')
pprint(summary['graph_stats'])

Loaded events: 6
Window count: 2
Graph stats from training summary:
{'mean_edge_observations_per_window': 3,
 'mean_edges_per_window': 1,
 'mean_nodes_per_window': 2,
 'num_windows': 2.0}


In [3]:
detector = load_detector(model_path)
threshold = summary['training_summary']['learned_threshold']
flagged_windows = detect_anomalies(detector, artifact_dataset.artifacts, threshold=threshold)

alert_payloads = []
for snapshot_index, (snapshot, flagged_scores) in enumerate(zip(snapshot_dataset.snapshots, flagged_windows, strict=True)):
    if not flagged_scores:
        continue
    candidate = extract_candidate_subgraph(
        snapshot=snapshot,
        flagged_scores=flagged_scores,
        alert_id=f'notebook-alert-{snapshot_index + 1:03d}',
    )
    alert_payloads.append(serialize_alert_subgraph(snapshot, candidate))

result = {
    'threshold': threshold,
    'flagged_windows': [len(window_scores) for window_scores in flagged_windows],
    'alerts': alert_payloads,
}
result

{'threshold': 3.4657359027997265,
 'flagged_windows': [0, 1],
 'alerts': [{'alert_id': 'notebook-alert-002',
   'window_start': '2026-01-01T12:15:00+00:00',
   'window_end': '2026-01-01T12:30:00+00:00',
   'flagged_edge_count': 1,
   'total_edge_count': 1,
   'nodes': [{'id': 'proc-1', 'type': 'PROCESS', 'name': '/bin/bash'},
    {'id': 'sock-1', 'type': 'SOCKET', 'name': '10.0.0.1:443'}],
   'edges': [{'key': 'proc-1:CONNECT:sock-1',
     'src': 'proc-1',
     'dst': 'sock-1',
     'type': 'CONNECT',
     'count': 4}]}]}

## What The Pipeline Can Do So Far

- Load PIDSMaker-style CSV provenance data
- Build time-windowed graph snapshots and graph artifacts
- Load a trained benign-behavior detector artifact
- Score windows for anomalous aggregated edges
- Extract and serialize candidate alert subgraphs

What is still deferred:

- A real GNN or PyTorch Geometric training path
- Raw DARPA CDM parsing
- Real LLM report generation